In [37]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [38]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [39]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [40]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['ECOM', 'ECOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [41]:
realignment_df

,psku realignment master,parent material_code old,asm,channel,new parent_material code,deletion indicator?,psku old,psku new
1,726067_ALL_E-Commerce,726067_SMO CLS MSL 500G MILLET MT ECOM B2C,ALL,ECOM,730975_SMO CLS MSL 550G MILLET MT ECOM,None,726067,730975
8,726065_ALL_E-Commerce,726065_SMO VEG TWST 500G MILLET MT ECOM,ALL,ECOM,731006_SMO VEG TWST 550G MILLET MT ECOM,None,726065,731006
9,726066_ALL_E-Commerce,726066_SMO PEP TOM 500G MILLET MT ECOM,ALL,ECOM,731008_SMO PEP TOM 550G MILLET MT ECOM,None,726066,731008
10,718574_ALL_E-Commerce,718574_SMO MSL&COR 500g PCH,ALL,ECOM,731007_SMO MAS COR 550G MILLET MT ECOM,None,718574,731007
135,718588_ALL_E-Commerce,718588_SETWET HAIRGEL ULTIMATE HOLD 250ml JAR,ALL,ECOM,730277_SW SPORTS EXTREM 250ML MT ECOM NF,None,718588,730277
...,...,...,...,...,...,...,...,...
419,724886_ALL_E-Commerce,724886_LIVON SRDR 100ML X 2 COMBO,ALL,ECOM,718783_LIVON SERUM DRY HAIR 100ml CBD,None,724886,718783
420,731073_ALL_E-Commerce,731073_PA BABY COMBO OIL400+OIL200,ALL,ECOM,725610_PAR ADV BABY OIL 400 ML,None,731073,725610
427,725043_ALL_E-Commerce,725043_NHR NSSB 175ML BOT PARENT,ALL,ECOM,725000_NHR NSSA 175ML BOT PARENT,None,725043,725000
428,721061_ALL_E-Commerce,721061_PA ALOE LIGHT 150ML,ALL,ECOM,718617_PAR ADV ALOEVERA ENRICHED CN 150ml BTL,None,721061,718617


In [42]:
from tqdm import tqdm

def impute_missing_dates(
        df,
        freq='M',
        key=['CHAIN', 'PARENT_MATERIAL_CODE'], 
        date_col='MONTH_DATE',
        max_date='2026-12-31'
    ):

    impute_df = df.copy()
    impute_df[date_col] = pd.to_datetime(impute_df[date_col])
    impute_df['key'] = impute_df[key].astype(str).agg('_'.join, axis=1)
    min_dates_df = impute_df.groupby(
        'key', as_index=False
    )[date_col].min()

    def impute_missing_dates_key(key, min_date, max_date):
        df_imputed = pd.DataFrame(
            pd.date_range(min_date, max_date, freq=freq),
            columns=[date_col]
        )
        df_imputed['key'] = key
        return df_imputed
    
    outputs = Parallel(n_jobs=-1)(
        delayed(impute_missing_dates_key)(row['key'], row[date_col], max_date)
        for idx, row in tqdm(min_dates_df.iterrows())
    )
    df_full = pd.concat(outputs)
    df_full.reset_index(drop=True, inplace=True)

    df_full = df_full.merge(
        impute_df, on=['key', date_col], how='left',
    )

    return df_full

### Primary Actuals + Sec Plan Data

In [43]:
plan_actuals_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        WHEN MCM.chain = 'Meesho' THEN 'Meesho'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date > '2022-12-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

plan_actuals_df = pd.read_sql(
    plan_actuals_query,
    prod_conn
)

In [44]:
top_chains = ['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Myntra', 'Nykaa','Meesho']

In [45]:
plan_actuals_df['CHAIN'].unique()

array(['1MG', 'Amazon ARIPL', 'Amazon RK', 'Big Basket', 'City Mall',
       'Dealshare', 'EMAZING DEALS', 'FATEHPURIA HYGIENE', 'First Cry',
       'Flipkart Grocery', 'Flipkart National', 'Meesho', 'Myntra',
       'Nykaa', 'Purplle'], dtype=object)

In [46]:
plan_actuals_df.columns = plan_actuals_df.columns.str.lower()
plan_actuals_df['month_date'] = pd.to_datetime(plan_actuals_df['month_date'])

In [47]:
plan_actuals_df.duplicated(subset=['chain', 'parent_material_code', 'material_group_code', 'month_date']).sum()

0

In [48]:
plan_actuals_df = realign_pskus(plan_actuals_df.copy(), column='parent_material_code')

In [49]:
plan_actuals_df.duplicated(subset=['chain', 'parent_material_code', 'material_group_code', 'month_date']).sum()

6599

In [50]:
plan_actuals_df['month_date'] = plan_actuals_df['month_date'] + MonthEnd(0)

In [51]:
plan_actuals_df = plan_actuals_df[
    (plan_actuals_df['parent_material_code'] != 715096)
]

In [52]:
715096 in plan_actuals_df['parent_material_code'].values

False

In [53]:
plan_actuals_df = plan_actuals_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [54]:
plan_actuals_df.shape

(145162, 8)

In [55]:
plan_actuals_df = impute_missing_dates(
    plan_actuals_df.copy(),
    key=['chain', 'parent_material_code'],
    date_col='month_date'
)

10560it [00:02, 4784.32it/s]


In [56]:
cols = ['chain', 'parent_material_code', 'material_group_code']

plan_actuals_df[cols] = plan_actuals_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [57]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].fillna(0)

In [58]:
plan_actuals_df['parent_material_code'] = plan_actuals_df['parent_material_code'].astype(int)

In [59]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

96

In [60]:
plan_actuals_df.shape

(365621, 9)

In [61]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = plan_actuals_df[plan_actuals_df.duplicated(subset=['key', 'month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
plan_actuals_df = plan_actuals_df.drop(indices_to_drop)

# Verify no duplicates remain
print(plan_actuals_df.duplicated(subset=['key', 'month_date']).sum())

0


In [62]:
plan_actuals_df.shape

(365482, 9)

In [63]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if plan_actuals_df[col].min() < 0:
        print(col)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [64]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    plan_actuals_df[col] = plan_actuals_df[col].clip(lower=0)

In [65]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [66]:
plan_actuals_df.sort_values(['key', 'month_date'], inplace=True)

In [67]:
plan_actuals_df['Primary P3M'] = plan_actuals_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [68]:
others_df = plan_actuals_df[
    ~plan_actuals_df['chain'].isin(top_chains)
]
plan_actuals_df = plan_actuals_df[
    plan_actuals_df['chain'].isin(top_chains)
]

In [69]:
print(others_df['chain'].unique())
print(plan_actuals_df['chain'].unique())

['1MG' 'City Mall' 'Dealshare' 'EMAZING DEALS' 'FATEHPURIA HYGIENE'
 'First Cry' 'Purplle']
['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa']


In [70]:
# x = plan_actuals_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x.dtypes
# x['plan_actuals_val'] = x['sec_actuals_vol_rum']*x['Index Rate']/10**7
# x.groupby(['month_date'])['plan_actuals_val'].sum().reset_index()#[20:]

In [71]:
# qtr_ind_rate_df

In [72]:
# x['sec_actuals_vol_rum'].sum()

In [73]:
# plan_actuals_df.to_csv('plan_actuals_ok.csv')

In [74]:
plan_actuals_df['sec_actuals_vol_rum'].sum()

11681281.076999998

### Offtakes

In [75]:
offtakes_monthly_query = """
SELECT
    OTM.platform_name AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    OTM.month_date,
    SUM(vol_in_rum) AS vol_in_roum
FROM 
    dwh_ecommplatform_offtake OTM 
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON OTM.material_code = MM.material_code
WHERE
    OTM.platform_name NOT IN ('Blinkit', 'Swiggy', 'Zepto', 'Others', 'Amazon D2C', 'Flipkart D2C') AND
    OTM.month_date > '2022-12-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
""" 

offtakes_monthly_df = pd.read_sql(
    offtakes_monthly_query,
    prod_conn
)

In [76]:
offtakes_monthly_df.columns = offtakes_monthly_df.columns.str.lower()

In [77]:
offtakes_monthly_df['month_date'] = pd.to_datetime(offtakes_monthly_df['month_date'])
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [78]:
offtakes_monthly_df.rename(columns={'vol_in_roum': 'offtake_vol_rum'}, inplace=True)

In [79]:
offtakes_monthly_df

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum
0,Amazon,710542,PCNO(R),2024-09-30,0.450
1,Amazon,710542,PCNO(R),2024-10-31,0.540
2,Amazon,718288,SAFF GOLD,2023-01-31,9.640
3,Amazon,718288,SAFF GOLD,2023-02-28,7.915
4,Amazon,718288,SAFF GOLD,2023-03-31,9.505
...,...,...,...,...,...
57599,Purplle,810673,PA_ESS_HO,2026-03-31,0.056
57600,Purplle,810673,PA_ESS_HO,2026-04-30,0.070
57601,Purplle,810673,PA_ESS_HO,2026-05-31,0.098
57602,Purplle,810674,PA_ESS_HO,2026-02-28,0.224


In [80]:
offtakes_monthly_df['month_date'].max()

Timestamp('2026-05-31 00:00:00')

In [81]:
mmonth_df = pd.read_sql("""
    SELECT * 
    FROM 
        TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_NEW
    WHERE
        month_date = '2026-06-30' AND
        run_month='2026-07-31'
""", 
    dev_conn
)

In [82]:
mmonth_df.columns = mmonth_df.columns.str.lower()

In [83]:
mmonth_df['month_date'] = pd.to_datetime(mmonth_df['month_date'])
mmonth_df['parent_material_code'] = mmonth_df['parent_material_code'].astype(int)

In [84]:
mmonth_df.head()

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,big_billion_days_lead_1,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month
0,2026-06-30,Amazon ARIPL,718288,29.418,40.851382,SAFF GOLD,0,0,0,0,0,0,0,0,0,0,0,1.311703,2,2026-07-31
1,2026-06-30,Amazon ARIPL,718312,0.321,1.121170,PCNO(R),0,0,0,0,0,0,0,0,0,0,0,NaN,2,2026-07-31
2,2026-06-30,Amazon ARIPL,718321,0.000,0.000000,SAFF KO,1,0,0,0,0,0,0,0,0,0,0,0.126506,2,2026-07-31
3,2026-06-30,Amazon ARIPL,718322,11.975,20.217097,SAFF KO,0,0,0,0,0,0,0,0,0,0,0,0.866589,2,2026-07-31
4,2026-06-30,Amazon ARIPL,718323,0.000,0.000000,SF_IMV_MK,1,0,0,0,0,0,0,0,0,0,0,0.000000,2,2026-07-31


In [85]:
mmonth_df = mmonth_df.groupby(
    ['platform_name', 'parent_material_code', 'brand_code', 'month_date'],
    as_index=False
)['vol_in_rum'].sum()

In [86]:
mmonth_df.rename(
    columns = {'platform_name': 'chain', 'brand_code': 'material_group_code', 'vol_in_rum': 'offtake_vol_rum'},
    inplace=True
)

In [87]:
offtakes_monthly_df = pd.concat([
    offtakes_monthly_df,
    mmonth_df
])

In [88]:
offtakes_monthly_df[offtakes_monthly_df['month_date'] == '2026-02-28']['offtake_vol_rum'].sum()

351669.82654000004

In [89]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

0

In [90]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.head()

,brand_code,portfolio
0,ADV-AHO-R,Hair Oils
1,ADV-COL-R,Hair Oils
2,ADV-COL-S,Hair Oils
3,BD_BDOL_M,Male Grooming
4,BD_HRWX_M,Male Grooming


In [91]:
len_before_merge = len(offtakes_monthly_df)
offtakes_monthly_df = offtakes_monthly_df.merge(
    brand_md_df.rename(columns={'brand_code': 'material_group_code'}),
    on=['material_group_code'],
    how='left'
)
assert len_before_merge == len(offtakes_monthly_df)

In [92]:
offtakes_monthly_df[offtakes_monthly_df['portfolio'].isna()]

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum,portfolio
8066,Amazon,732778,SFOATS-PR,2026-02-28,0.6910,NaN
8067,Amazon,732778,SFOATS-PR,2026-03-31,1.0330,NaN
8068,Amazon,732778,SFOATS-PR,2026-04-30,1.6650,NaN
8069,Amazon,732778,SFOATS-PR,2026-05-31,2.4330,NaN
8070,Amazon,732779,SFOATS-PR,2026-03-31,0.0412,NaN
...,...,...,...,...,...,...
48677,Myntra,732940,PA_SHMP_R,2026-05-31,62.0000,NaN
48678,Myntra,732941,PA_SHMP_R,2026-05-31,29.9200,NaN
48679,Myntra,732942,PA_SHMP_R,2026-05-31,14.9500,NaN
50169,Myntra,811287,PA_RSW_SR,2026-05-31,4.9000,NaN


In [93]:
offtakes_monthly_df['chain'] = np.where(
    (offtakes_monthly_df['chain'] == 'Amazon'),
    np.where(
        offtakes_monthly_df['portfolio'].isin(['Saffola Oils', 'Foods']),
        'Amazon ARIPL',
        'Amazon RK'
    ),
    offtakes_monthly_df['chain']
)

In [94]:
offtakes_monthly_df[offtakes_monthly_df['chain'].str.startswith('Amazon')][['chain', 'portfolio']].drop_duplicates()

,chain,portfolio
0,Amazon RK,CNO
2,Amazon ARIPL,Saffola Oils
123,Amazon RK,Hair Oils
219,Amazon ARIPL,Foods
222,Amazon RK,Others
319,Amazon RK,Male Grooming
420,Amazon RK,Prem. Hair Nour.
1013,Amazon RK,Skin Care
8066,Amazon RK,NaN
57605,Amazon ARIPL,CNO


In [95]:
offtakes_monthly_df['chain'].unique()

array(['Amazon RK', 'Amazon ARIPL', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa', 'Purplle'],
      dtype=object)

In [96]:
offtakes_monthly_df['chain'] = offtakes_monthly_df['chain'].replace({
    'MYNTRA': 'Myntra',
    'BB': 'Big Basket',
    'Flipkart Mintues': 'Flipkart National',
    'Flipkart Minutes': 'Flipkart National'
})

In [97]:
del offtakes_monthly_df['portfolio']

In [98]:
offtakes_monthly_df = realign_pskus(offtakes_monthly_df.copy(), 'parent_material_code')

In [99]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

1362

In [100]:
offtakes_monthly_df = offtakes_monthly_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'],
    as_index=False
).sum()

In [101]:
offtakes_monthly_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

4

In [102]:
offtakes_monthly_df.head()

,chain,parent_material_code,material_group_code,month_date,offtake_vol_rum
0,Amazon ARIPL,718288,SAFF GOLD,2023-01-31,9.640
1,Amazon ARIPL,718288,SAFF GOLD,2023-02-28,7.915
2,Amazon ARIPL,718288,SAFF GOLD,2023-03-31,9.505
3,Amazon ARIPL,718288,SAFF GOLD,2023-04-30,9.290
4,Amazon ARIPL,718288,SAFF GOLD,2023-05-31,8.050


In [103]:
offtakes_monthly_df = impute_missing_dates(
    offtakes_monthly_df.copy(),
    key=['chain', 'parent_material_code'],
    date_col='month_date'
)

3357it [00:00, 5922.97it/s]


In [104]:
offtakes_monthly_df.sort_values(by=['key', 'month_date'], inplace=True)

cols = ['chain', 'parent_material_code', 'material_group_code']

offtakes_monthly_df[cols] = offtakes_monthly_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [105]:
offtakes_monthly_df['run_month'] = offtakes_monthly_df['month_date'] + MonthEnd(0)

In [106]:
offtakes_monthly_df['offtake_vol_rum'] = offtakes_monthly_df['offtake_vol_rum'].fillna(0)

In [107]:
offtakes_monthly_df['offtake_vol_rum'].min()

0.0

In [108]:
offtakes_monthly_df['parent_material_code'] = offtakes_monthly_df['parent_material_code'].astype(int)

In [109]:
# offtakes_monthly_df['key'] = offtakes_monthly_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [110]:
(offtakes_monthly_df['key'] == offtakes_monthly_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

True

In [111]:
offtakes_monthly_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31
...,...,...,...,...,...,...,...
120118,2026-08-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-08-31
120119,2026-09-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-09-30
120120,2026-10-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-10-31
120121,2026-11-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-11-30


In [112]:
offtakes_monthly_df.duplicated(subset=['key', 'month_date']).sum()

4

In [113]:
715096 in offtakes_monthly_df['parent_material_code'].values

True

In [114]:
offtakes_monthly_df = offtakes_monthly_df[
    offtakes_monthly_df['parent_material_code'] != 715096
]

In [115]:
offtakes_monthly_df['offtake_vol_rum'].min()

0.0

In [116]:
offtakes_monthly_df.to_excel('Offtakes_Chain_PSKU_ECOM.xlsx', index=False)

In [117]:
offtakes_monthly_df.groupby(['month_date'])['offtake_vol_rum'].sum()[30:]

month_date
2025-07-31    278377.263560
2025-08-31    258971.933770
2025-09-30    544883.557660
2025-10-31    523456.543610
2025-11-30    483400.337140
2025-12-31    496069.679560
2026-01-31    450861.594850
2026-02-28    351669.826540
2026-03-31    366449.897800
2026-04-30    328414.346900
2026-05-31    380867.395930
2026-06-30    285294.761216
2026-07-31         0.000000
2026-08-31         0.000000
2026-09-30         0.000000
2026-10-31         0.000000
2026-11-30         0.000000
2026-12-31         0.000000
Name: offtake_vol_rum, dtype: float64

In [118]:
offtakes_monthly_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31
...,...,...,...,...,...,...,...
120118,2026-08-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-08-31
120119,2026-09-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-09-30
120120,2026-10-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-10-31
120121,2026-11-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-11-30


In [119]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate
qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [120]:
qtr_ind_rate_df

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.15200
1,2027-03-31,TRU_RAWDF,800.00000
2,2027-03-31,TRU_PDRFR,850.57000
3,2027-03-31,TRU_OATS,177.07000
4,2027-03-31,TRU_QUINO,204.75000
...,...,...,...
349,2018-03-31,PADV-HRAHF,0.00000
350,2018-03-31,PAR T HFS,0.00000
351,2020-03-31,LIVON HG Shampoo,0.00000
352,2020-03-31,NHO ACE,0.00000


In [121]:
# x = offtakes_monthly_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x['offtake_val'] = x['offtake_vol_rum']*x['Index Rate']/10**7
# x.groupby(['month_date'])['offtake_val'].sum().reset_index()[20:]

### SOH

In [122]:
soh_df = pd.read_csv('/data/aman_singh/acuuracy_check/soh_base_jul_run.csv')

In [123]:
soh_df.columns = soh_df.columns.str.lower()

In [124]:
soh_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,roum,roum divide,month,day,month.1,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,unnamed: 41
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89874,Amazon RK,B0GRVG1BH2,969.0,811241,PA ROSMARY HAIR SRAY 100ML,PA_RSW_SR,L,100.0,96.900000,740.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89875,Amazon RK,B0GT8SRWNG,69.0,811199,PA PEPPERMINT ESS OIL 14ML,PA_ESS_HO,L,14.0,0.966000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89876,Amazon RK,B0GT8ZGZ4T,172.0,811197,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,L,14.0,2.408000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89877,Amazon RK,B0GT91H554,118.0,811198,PA SANDALWOOD ESS OIL 14ML,PA_ESS_HO,L,14.0,1.652000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [125]:
soh_df.groupby(['chain'])['date'].max()

chain
Amazon ARIPL         2026-07-01
Amazon RK            2026-07-01
BB                   2025-03-22
Big Basket           2026-07-01
Blinkit              2026-07-01
Flipkart Grocery     2026-06-01
Flipkart Mintues     2025-10-31
Flipkart Minutes     2026-07-01
Flipkart National    2026-07-01
Flipkart-Grocery     2026-07-01
Meesho               2026-07-01
Myntra               2026-07-01
Nykaa                2026-07-01
Purplle              2025-11-17
Swiggy               2026-07-01
Zepto                2026-07-01
Name: date, dtype: object

In [126]:
import numpy as np
soh_df['date'] = np.where(
    soh_df['date'] > '2026-06-30',
    '2026-06-30',
    soh_df['date']
)

In [127]:
soh_df.groupby(['chain'])['date'].max()

chain
Amazon ARIPL         2026-06-30
Amazon RK            2026-06-30
BB                   2025-03-22
Big Basket           2026-06-30
Blinkit              2026-06-30
Flipkart Grocery     2026-06-01
Flipkart Mintues     2025-10-31
Flipkart Minutes     2026-06-30
Flipkart National    2026-06-30
Flipkart-Grocery     2026-06-30
Meesho               2026-06-30
Myntra               2026-06-30
Nykaa                2026-06-30
Purplle              2025-11-17
Swiggy               2026-06-30
Zepto                2026-06-30
Name: date, dtype: object

In [128]:
material_master = pd.read_sql("""
SELECT * 
FROM
    mst_material
WHERE
    company_code='MIL' AND
    latest_record_ind=1
""",
    prod_conn
)

In [129]:
material_master.columns = material_master.columns.str.lower()
material_master['material_code'] = material_master['material_code'].astype(int)

In [130]:
soh_df.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1', 'unnamed: 18', 'unnamed: 19',
       'unnamed: 20', 'unnamed: 21', 'unnamed: 41'],
      dtype='object')

In [131]:
material_master.columns

Index(['company_code', 'material_code', 'material_desc', 'material_group_code',
       'material_group_desc', 'division_code', 'division_name', 'uom_base',
       'uom_weight', 'gross_weight', 'net_weight', 'parent_material_desc',
       'material_type', 'material_type_desc', 'uom_sales', 'uom_reporting',
       'mg1_code', 'mg1_desc', 'mg2_code', 'mg2_desc', 'mg3_code', 'mg3_desc',
       'mg4_code', 'mg4_desc', 'mg5_code', 'mg5_desc', 'profit_centre_code',
       'vol_per_unit', 'unit_per_case_nbr', 'convert_to_ton', 'convert_to_kl',
       'convert_to_l', 'convert_to_kg', 'convert_to_ml', 'convert_to_gm',
       'csd_id', 'ean_id', 'standard_cost_amt', 'source_system_id',
       'last_bi_updt_date', 'last_src_updt_date', 'material_sk',
       'effective_start_date', 'effective_end_date', 'latest_record_ind',
       'version_nbr', 'parent_material_code', 'parent_material1_code',
       'parent_material1_desc', 'shelf_life_days',
       'manual_material_group_desc', 'manual_mg1_desc',

In [132]:
for sku in soh_df['material_code'].unique():
    try:
        int(sku)
    except:
        print(sku)

Combo


In [133]:
soh_df = soh_df[soh_df['material_code'] != 'Combo']

In [134]:
soh_df['material_code'] = soh_df['material_code'].astype(int)

In [135]:
soh_df.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1', 'unnamed: 18', 'unnamed: 19',
       'unnamed: 20', 'unnamed: 21', 'unnamed: 41'],
      dtype='object')

In [136]:
soh_df.drop(['material_group_code', 'vol_per_unit', 'uom_reporting'], axis=1, inplace=True)

In [137]:
len_before_merge = len(soh_df)
soh_df = soh_df.merge(
    material_master[['material_code', 'parent_material_code', 'material_group_code', 'vol_per_unit', 'uom_reporting']],
    on=['material_code'],
    how='left'
)
assert len_before_merge == len(soh_df)
del len_before_merge

In [138]:
# soh_df[soh_df['parent_material_code'].isna()]

In [139]:
soh_df = soh_df[soh_df['parent_material_code'].notna()]

In [140]:
soh_df['parent_material_code'] = soh_df['parent_material_code'].astype(int)

In [141]:
soh_df['chain'] = soh_df['chain'].replace({
    'MYNTRA': 'Myntra',
    'BB': 'Big Basket',
    'Flipkart Mintues': 'Flipkart National',
    'Flipkart Minutes': 'Flipkart National'
})

In [142]:
sorted(soh_df['chain'].unique())

['Amazon ARIPL',
 'Amazon RK',
 'Big Basket',
 'Blinkit',
 'Flipkart Grocery',
 'Flipkart National',
 'Flipkart-Grocery',
 'Meesho',
 'Myntra',
 'Nykaa',
 'Purplle',
 'Swiggy',
 'Zepto']

In [143]:
soh_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,month.1,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,unnamed: 41,parent_material_code,material_group_code,vol_per_unit,uom_reporting
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,718322,SAFF KO,5000.0000,KL
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,718492,SFOATS-FL,37.9997,TO
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,718494,SFOATS-FL,37.9997,TO
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,807029,SAFF SALT,1000.0000,TO
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,718328,SAFF KOCO,1000.0000,KL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89869,Amazon RK,B0GRVG1BH2,969.0,811241,PA ROSMARY HAIR SRAY 100ML,PA_RSW_SR,L,100.0,96.900000,740.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,811287,PA_RSW_SR,100.0000,L
89870,Amazon RK,B0GT8SRWNG,69.0,811199,PA PEPPERMINT ESS OIL 14ML,PA_ESS_HO,L,14.0,0.966000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,811267,PA_ESS_HO,14.0000,L
89871,Amazon RK,B0GT8ZGZ4T,172.0,811197,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,L,14.0,2.408000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,811269,PA_ESS_HO,14.0000,L
89872,Amazon RK,B0GT91H554,118.0,811198,PA SANDALWOOD ESS OIL 14ML,PA_ESS_HO,L,14.0,1.652000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,811268,PA_ESS_HO,14.0000,L


In [144]:
soh_df['current_soh'] = soh_df['soh'] * soh_df['vol_per_unit'] / soh_df['uom_reporting'].map(
    lambda x: (10 ** 6) if x in ('KL', 'TO') else (10 ** 3)
)

In [145]:
soh_df.dtypes

chain                    object
fsn                      object
soh                     float64
material_code             int64
desc                     object
brand                    object
uom                      object
vol per unit            float64
vol                     float64
fy index                float64
bpm in lacs             float64
category                 object
ecom brands              object
file                     object
vol in kl               float64
psku                     object
pdes                     object
date                     object
asin                     object
asin.1                   object
platform_name.1          object
product_title            object
ean                     float64
units                   float64
vol_in_lit              float64
vol in rum              float64
indexrate               float64
indexbpm                float64
club sku                 object
roum                    float64
roum divide             float64
month   

In [146]:
soh_df.rename(
    columns={'date': 'inv date'},
    inplace=True
)

In [147]:
soh_df = realign_pskus(soh_df.copy(), 'parent_material_code')

In [148]:
soh_df = soh_df.groupby(
    ['chain', 'parent_material_code', 'inv date'], as_index=False
)['current_soh'].sum().rename(columns={
    'inv date': 'as_on_date'
})

In [149]:
soh_df['key'] = soh_df[['chain', 'parent_material_code']].astype(str).agg('_'.join, axis=1)

In [150]:
soh_df['as_on_date'] = pd.to_datetime(soh_df['as_on_date'])

In [151]:
soh_df['run_month'] = soh_df['as_on_date'] + MonthEnd(0)

In [152]:
soh_df.dtypes

chain                           object
parent_material_code             int64
as_on_date              datetime64[ns]
current_soh                    float64
key                             object
run_month               datetime64[ns]
dtype: object

In [153]:
soh_df

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
...,...,...,...,...,...,...
42235,Zepto,811181,2026-06-01,0.216,Zepto_811181,2026-06-30
42236,Zepto,811181,2026-06-30,0.325,Zepto_811181,2026-06-30
42237,Zepto,811287,2026-05-01,9.500,Zepto_811287,2026-05-31
42238,Zepto,811287,2026-06-01,55.600,Zepto_811287,2026-06-30


In [154]:
date_wise_soh_vol_sum = soh_df.groupby(['as_on_date'], as_index=False)['current_soh'].sum()

In [155]:
date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]

,as_on_date,current_soh


In [156]:
soh_df[
    soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]['current_soh'].sum()

0.0

In [157]:
soh_df = soh_df[
    ~soh_df['as_on_date'].isin(
        date_wise_soh_vol_sum[date_wise_soh_vol_sum['current_soh'] == 0]['as_on_date'].unique()
    )
]

In [158]:
715096 in soh_df['parent_material_code'].values

False

In [159]:
soh_df[['chain', 'as_on_date']].drop_duplicates().sort_values(by=['chain', 'as_on_date'])

,chain,as_on_date
0,Amazon ARIPL,2024-12-14
1,Amazon ARIPL,2025-01-25
2,Amazon ARIPL,2025-02-22
3,Amazon ARIPL,2025-03-29
4,Amazon ARIPL,2025-04-19
...,...,...
38323,Zepto,2026-02-28
38324,Zepto,2026-04-01
38304,Zepto,2026-05-01
38305,Zepto,2026-06-01


In [160]:
soh_df[soh_df['chain'] == 'Amazon ARIPL'].head(60)

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
5,Amazon ARIPL,718288,2025-05-31,12.048,Amazon ARIPL_718288,2025-05-31
6,Amazon ARIPL,718288,2025-06-19,9.138,Amazon ARIPL_718288,2025-06-30
7,Amazon ARIPL,718288,2025-07-31,6.840,Amazon ARIPL_718288,2025-07-31
8,Amazon ARIPL,718288,2025-08-02,17.118,Amazon ARIPL_718288,2025-08-31
9,Amazon ARIPL,718288,2025-10-05,18.864,Amazon ARIPL_718288,2025-10-31


### Forecasts

In [161]:
forecasts = pd.read_excel(
    r"/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_july_live.xlsx",
    sheet_name='base'
)
forecasts = forecasts[forecasts['key'].notna()]
forecasts.head()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,pred_value_SARIMA,shrink_ratio_sarima,recency_heuristic_sarima,seasonal_heuristic_sarima,non_seasonal_heuristic_sarima,final_heuristic_sarima,final_heuristic_sarima_value,final_heuristic_sarima_value_2,ratio_last_year,quarter
0,Amazon ARIPL_718288,2026-07-31,23.908,25.586,24.234557,22.6834,0.331999,0.355301,0.336534,0.314994,...,0.296432,1.058293,22.340048,25.670285,20.927831,22.340048,0.310226,0.345424,1.443409,3
1,Amazon ARIPL_718288,2026-08-31,23.908,25.586,26.673760,29.3248,0.331999,0.355301,0.370406,0.407220,...,0.438793,0.869838,31.598475,31.598475,29.502677,31.598475,0.438793,0.380843,1.074137,3
2,Amazon ARIPL_718288,2026-09-30,23.908,25.586,27.384169,23.8084,0.331999,0.355301,0.380271,0.330616,...,0.364174,0.954803,26.225007,26.225007,24.146537,26.225007,0.364174,0.391614,1.087573,3
3,Amazon ARIPL_718288,2026-10-31,23.908,25.586,31.024002,26.9124,0.331999,0.355301,0.430816,0.373720,...,0.381100,0.933360,27.443841,28.718123,23.858217,27.443841,0.381100,0.439289,1.266715,4
4,Amazon ARIPL_718288,2026-11-30,23.908,25.586,28.098806,30.1308,0.331999,0.355301,0.390195,0.418412,...,0.420347,0.888719,30.270134,30.270134,27.525006,30.270134,0.420347,0.401523,1.184903,4


In [162]:
forecasts['parent_material_code'] = forecasts['parent_material_code'].astype(int)

In [163]:
forecasts.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf',
       ...
       'pred_value_SARIMA', 'shrink_ratio_sarima', 'recency_heuristic_sarima',
       'seasonal_heuristic_sarima', 'non_seasonal_heuristic_sarima',
       'final_heuristic_sarima', 'final_heuristic_sarima_value',
       'final_heuristic_sarima_value_2', 'ratio_last_year', 'quarter'],
      dtype='object', length=162)

In [164]:
forecasts['Final Heuristic Prophet Vol'] = forecasts['final Heuristic Value'] * (10 ** 7) / forecasts['qtr_ind_rate_x']
forecasts.rename(columns = {'run_month_x': 'run_month'}, inplace=True)

In [165]:
forecasts = forecasts[['key', 'month_date', 'platform_name', 'parent_material_code', 'brand_code', 
                       'run_month', 'M month', 'Final Heuristic Prophet Vol', 'skipped']]

In [166]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-07-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [167]:
forecasts.duplicated(['key', 'run_month', 'month_date']).sum()

0

In [168]:
forecasts[forecasts.duplicated(['key', 'run_month', 'month_date'])]

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped


In [169]:
forecasts['platform_name'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa'], dtype=object)

In [170]:
# forecasts.rename(columns={'Final Heuristic Prophet 2 Vol': 'Final Heuristic Prophet Vol'}, inplace=True)

In [171]:
top_chains

['Amazon ARIPL',
 'Amazon RK',
 'Big Basket',
 'Flipkart Grocery',
 'Flipkart National',
 'Myntra',
 'Nykaa',
 'Meesho']

### Collate Everything

In [172]:
print(f"SOH", soh_df['chain'].unique())
print("Offtakes:", offtakes_monthly_df['chain'].unique())
print("Plan Actuals:", plan_actuals_df['chain'].unique())

SOH ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Blinkit' 'Flipkart Grocery'
 'Flipkart National' 'Flipkart-Grocery' 'Meesho' 'Myntra' 'Nykaa'
 'Purplle' 'Swiggy' 'Zepto']
Offtakes: ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa' 'Purplle']
Plan Actuals: ['Amazon ARIPL' 'Amazon RK' 'Big Basket' 'Flipkart Grocery'
 'Flipkart National' 'Meesho' 'Myntra' 'Nykaa']


In [173]:
offtakes_monthly_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31
...,...,...,...,...,...,...,...
120118,2026-08-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-08-31
120119,2026-09-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-09-30
120120,2026-10-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-10-31
120121,2026-11-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-11-30


In [174]:
final_df = plan_actuals_df[
    ['key', 'chain', 'parent_material_code', 'material_group_code']
].drop_duplicates()

In [175]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = final_df[final_df.duplicated(subset=['key'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
final_df = final_df.drop(indices_to_drop)

# Verify no duplicates remain
print(final_df.duplicated(subset=['key']).sum())

0


In [176]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-07-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [177]:
tmp_df = pd.DataFrame()

for rm in ['2026-07-31']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_df.copy()
        tmp_df2['run_month'] = pd.to_datetime(rm)
        tmp_df2['month_date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_df = tmp_df.copy()    
del tmp_df

### Merge Plan

In [178]:
plan_actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [179]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    plan_actuals_df[['key', 'month_date', 'pri_actuals_vol_rum', 'sec_apo_plan_vol_rum', 'Primary P3M']],
    on=['key', 'month_date'],
    how='left'
)
assert len(final_df) == len_before_merge
del len_before_merge

In [180]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0
1,Amazon ARIPL_715099,Amazon ARIPL,715099,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0
2,Amazon ARIPL_715100,Amazon ARIPL,715100,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0
3,Amazon ARIPL_715106,Amazon ARIPL,715106,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0
4,Amazon ARIPL_715107,Amazon ARIPL,715107,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
60485,Nykaa_811268,Nykaa,811268,PA_ESS_HO,2026-07-31,2027-03-31,NaN,NaN,NaN
60486,Nykaa_811269,Nykaa,811269,PA_ESS_HO,2026-07-31,2027-03-31,NaN,NaN,NaN
60487,Nykaa_811279,Nykaa,811279,SAF_CDPRS,2026-07-31,2027-03-31,NaN,NaN,NaN
60488,Nykaa_811287,Nykaa,811287,PA_RSW_SR,2026-07-31,2027-03-31,NaN,NaN,NaN


In [181]:
plan_actuals_df[plan_actuals_df['month_date']<'2026-06-30'].to_csv('/data/aman_singh/acuuracy_check/plan_actuals_df_ecom.csv', index=False)

In [182]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [183]:
for col in ['Primary P3M']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

### Merge offtakes

In [184]:
final_df['key'].nunique()

6049

In [185]:
offtakes_monthly_df['key'].nunique()

3356

In [186]:
yy = final_df.copy()

In [187]:
### Monthly Actual Offtakes
len_before_merge = len(final_df)
final_df = final_df.merge(
    offtakes_monthly_df[['month_date', 'key', 'offtake_vol_rum']].rename(columns={
        'offtake_vol_rum': 'Offtake Chain PSKU'
    }),
    on=['month_date', 'key'],
    how='left'
)   
assert len_before_merge == len(final_df)
del len_before_merge

In [188]:
final_df[final_df['month_date'] == '2026-05-31']['key'].nunique()

0

In [189]:
# x = final_df.copy()
# x.rename(columns = {'material_group_code':'Brand'},inplace = True)
# len_before_merge = len(x)
# x = x.merge(
#     qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
#         'brand_code': 'Brand',
#         'qtr_ind_rate': 'Index Rate'
#     }),
#     on=['Brand'],
#     how='left'
# )
# assert len_before_merge == len(x)
# del len_before_merge
# x['offtake_val'] = x['Offtake Chain PSKU']*x['Index Rate']/10**7
# x.groupby(['month_date'])['offtake_val'].sum().reset_index()#[20:]

In [190]:
# lits = []
# for k in offtakes_monthly_df[offtakes_monthly_df['month_date'] == '2026-05-31']['key'].unique():
#     if(k not in final_df['key'].unique()):
#         lits.append(k)

In [191]:
# lits

In [192]:
# len(lits)

In [193]:
# final_df[final_df['Offtake Chain PSKU'].isna()]

In [194]:
mappings = {}

for run_month in final_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-07-31 00:00:00'): {Timestamp('2026-07-31 00:00:00'): 'M',
  Timestamp('2026-08-31 00:00:00'): 'M+1',
  Timestamp('2026-09-30 00:00:00'): 'M+2',
  Timestamp('2026-10-31 00:00:00'): 'M+3',
  Timestamp('2026-11-30 00:00:00'): 'M+4',
  Timestamp('2026-12-31 00:00:00'): 'M+5',
  Timestamp('2027-01-31 00:00:00'): 'M+6',
  Timestamp('2027-02-28 00:00:00'): 'M+7',
  Timestamp('2027-03-31 00:00:00'): 'M+8'}}

In [195]:
final_df['M month'] = final_df.apply(
    lambda x: mappings[x['run_month']].get(x['month_date'], np.nan),
    axis=1
)

In [196]:
final_df[['run_month', 'month_date', 'M month']].drop_duplicates()

,run_month,month_date,M month
0,2026-07-31,2026-06-30,NaN
1,2026-07-31,2026-07-31,M
2,2026-07-31,2026-08-31,M+1
3,2026-07-31,2026-09-30,M+2
4,2026-07-31,2026-10-31,M+3
5,2026-07-31,2026-11-30,M+4
6,2026-07-31,2026-12-31,M+5
7,2026-07-31,2027-01-31,M+6
8,2026-07-31,2027-02-28,M+7
9,2026-07-31,2027-03-31,M+8


### Merge Forecasts

In [197]:
forecasts.head()

,key,month_date,platform_name,parent_material_code,brand_code,run_month,M month,Final Heuristic Prophet Vol,skipped
0,Amazon ARIPL_718288,2026-07-31,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,M,22.683400,0
1,Amazon ARIPL_718288,2026-08-31,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,M+1,27.425380,0
2,Amazon ARIPL_718288,2026-09-30,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,M+2,23.808400,0
3,Amazon ARIPL_718288,2026-10-31,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,M+3,31.634215,0
4,Amazon ARIPL_718288,2026-11-30,Amazon ARIPL,718288,SAFF GOLD,2026-07-31,M+4,28.914594,0


In [198]:
forecasts.duplicated(subset=['key', 'month_date', 'run_month']).sum()

0

In [199]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    forecasts[['key', 'month_date', 'run_month', 'Final Heuristic Prophet Vol']],
    on=['key', 'month_date', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [200]:
final_df.rename(columns={'Final Heuristic Prophet Vol': 'Offtake Chain PSKU Forecast Vol'}, inplace=True)

### Norms

In [201]:
soh_df['as_on_date'].min()

Timestamp('2024-12-03 00:00:00')

In [202]:
soh_df[~soh_df['chain'].isin(top_chains)]['chain'].unique()

array(['Blinkit', 'Flipkart-Grocery', 'Purplle', 'Swiggy', 'Zepto'],
      dtype=object)

In [203]:
soh_df

,chain,parent_material_code,as_on_date,current_soh,key,run_month
0,Amazon ARIPL,718288,2024-12-14,16.794,Amazon ARIPL_718288,2024-12-31
1,Amazon ARIPL,718288,2025-01-25,28.038,Amazon ARIPL_718288,2025-01-31
2,Amazon ARIPL,718288,2025-02-22,23.850,Amazon ARIPL_718288,2025-02-28
3,Amazon ARIPL,718288,2025-03-29,9.204,Amazon ARIPL_718288,2025-03-31
4,Amazon ARIPL,718288,2025-04-19,7.044,Amazon ARIPL_718288,2025-04-30
...,...,...,...,...,...,...
42235,Zepto,811181,2026-06-01,0.216,Zepto_811181,2026-06-30
42236,Zepto,811181,2026-06-30,0.325,Zepto_811181,2026-06-30
42237,Zepto,811287,2026-05-01,9.500,Zepto_811287,2026-05-31
42238,Zepto,811287,2026-06-01,55.600,Zepto_811287,2026-06-30


In [204]:
avg_inventory = soh_df[
    (soh_df['chain'].isin(top_chains))
].groupby(
    ['key'], as_index=False
)['current_soh'].mean()

In [205]:
soh_df['as_on_date'].min(), soh_df['as_on_date'].max()

(Timestamp('2024-12-03 00:00:00'), Timestamp('2026-06-30 00:00:00'))

In [206]:
avg_daily_offtakes = offtakes_monthly_df[
    (offtakes_monthly_df['month_date'] > '2024-11-30') &
    (offtakes_monthly_df['month_date'] < '2026-07-01')
].copy()
avg_daily_offtakes['days'] = avg_daily_offtakes['month_date'].dt.day
avg_daily_offtakes = avg_daily_offtakes.groupby(
    ['key'], as_index=False
)[['offtake_vol_rum', 'days']].sum()

In [207]:
avg_daily_offtakes

,key,offtake_vol_rum,days
0,Amazon ARIPL_718288,368.0100,577
1,Amazon ARIPL_718312,0.3210,30
2,Amazon ARIPL_718321,0.0000,577
3,Amazon ARIPL_718322,141.4050,577
4,Amazon ARIPL_718323,0.0000,577
...,...,...,...
3351,Purplle_807725,0.0000,577
3352,Purplle_809042,1.8000,365
3353,Purplle_809250,0.0204,577
3354,Purplle_810673,0.3500,150


In [208]:
avg_daily_offtakes['avg_offtakes_vol_rum'] = avg_daily_offtakes['offtake_vol_rum'] / avg_daily_offtakes['days']

In [209]:
norm_days = avg_inventory.merge(
    avg_daily_offtakes[['key', 'avg_offtakes_vol_rum']],
    on=['key'],
    how='left'
) 
assert len(norm_days) == len(avg_inventory)

In [210]:
norm_days

,key,current_soh,avg_offtakes_vol_rum
0,Amazon ARIPL_718288,21.281053,0.637799
1,Amazon ARIPL_718312,0.055778,0.010700
2,Amazon ARIPL_718322,11.557105,0.245069
3,Amazon ARIPL_718328,4.159737,0.128939
4,Amazon ARIPL_718330,3.891842,0.123847
...,...,...,...
2628,Nykaa_810673,2.097200,0.056461
2629,Nykaa_810674,1.119067,0.018411
2630,Nykaa_811005,1.512000,NaN
2631,Nykaa_811019,43.949143,0.505044


In [211]:
norm_days['norm_days'] = norm_days['current_soh'] / norm_days['avg_offtakes_vol_rum']

In [212]:
# norm_days['norm_days'] = np.minimum(np.maximum(norm_days['norm_days'] - 30, 0), 30)

norm_days['norm_days'] = np.maximum(np.minimum(norm_days['norm_days'], 30), 5)

In [213]:
norm_days['norm_days'] = norm_days['norm_days'].fillna(5)

In [214]:
norms = final_df.copy()

In [215]:
norms = norms[['key', 'run_month', 'month_date', 'Offtake Chain PSKU Forecast Vol']]

In [216]:
norms['total_days_in_month'] = norms['month_date'].dt.day

In [217]:
norms.isnull().sum()

key                                    0
run_month                              0
month_date                             0
Offtake Chain PSKU Forecast Vol    37497
total_days_in_month                    0
dtype: int64

In [218]:
norms

,key,run_month,month_date,Offtake Chain PSKU Forecast Vol,total_days_in_month
0,Amazon ARIPL_715098,2026-07-31,2026-06-30,NaN,30
1,Amazon ARIPL_715098,2026-07-31,2026-07-31,NaN,31
2,Amazon ARIPL_715098,2026-07-31,2026-08-31,NaN,31
3,Amazon ARIPL_715098,2026-07-31,2026-09-30,NaN,30
4,Amazon ARIPL_715098,2026-07-31,2026-10-31,NaN,31
...,...,...,...,...,...
60485,Nykaa_811416,2026-07-31,2026-11-30,NaN,30
60486,Nykaa_811416,2026-07-31,2026-12-31,NaN,31
60487,Nykaa_811416,2026-07-31,2027-01-31,NaN,31
60488,Nykaa_811416,2026-07-31,2027-02-28,NaN,28


In [219]:
norms.isnull().sum()

key                                    0
run_month                              0
month_date                             0
Offtake Chain PSKU Forecast Vol    37497
total_days_in_month                    0
dtype: int64

In [220]:
len_before_merge = len(norms)
norms = norms.merge(
    norm_days[['key', 'norm_days']],
    on=['key'],
    how='left'
)
assert len_before_merge == len(norms)
del len_before_merge

In [221]:
norms['safety_stock'] = norms['Offtake Chain PSKU Forecast Vol'] *  norms['norm_days'] / norms['total_days_in_month']

In [222]:
norm_days[norm_days.duplicated(subset=['key'])]

,key,current_soh,avg_offtakes_vol_rum,norm_days


In [223]:
final_df[final_df['key'] == 'Amazon ARIPL_715098']

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,M month,Offtake Chain PSKU Forecast Vol
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-07-31,0.0,0.0,0.0,NaN,M,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-08-31,0.0,0.0,0.0,NaN,M+1,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-09-30,0.0,0.0,0.0,NaN,M+2,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-10-31,0.0,0.0,0.0,NaN,M+3,NaN
5,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-11-30,0.0,0.0,0.0,NaN,M+4,NaN
6,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-12-31,0.0,0.0,0.0,NaN,M+5,NaN
7,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2027-01-31,NaN,NaN,0.0,NaN,M+6,NaN
8,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2027-02-28,NaN,NaN,0.0,NaN,M+7,NaN
9,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2027-03-31,NaN,NaN,0.0,NaN,M+8,NaN


In [224]:
final_df[final_df.duplicated(subset=['key', 'run_month', 'month_date'])]#['material_group_code'].unique()

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,M month,Offtake Chain PSKU Forecast Vol


In [225]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain PSKU Forecast Vol', 'total_days_in_month', 'norm_days'], axis=1).rename(columns={
        'safety_stock': 'norms_soh'
    }),
    on=['key', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [226]:
norms['month_date'] = norms['month_date'] - MonthEnd(1)

In [227]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    norms.drop(['Offtake Chain PSKU Forecast Vol', 'total_days_in_month'], axis=1),
    on=['key', 'run_month', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [228]:
final_df.sort_values(
    by=['run_month', 'material_group_code', 'key', 'month_date'], inplace=True
)

In [229]:
final_df['safety_stock'] = final_df['safety_stock'].fillna(0)

In [230]:
chain_wise_max_soh_dates = soh_df[soh_df['chain'].isin(top_chains)].groupby(
    ['run_month', 'chain'], as_index=False
)['as_on_date'].max()

chain_wise_max_soh_dates

,run_month,chain,as_on_date
0,2024-12-31,Amazon ARIPL,2024-12-14
1,2024-12-31,Amazon RK,2024-12-19
2,2024-12-31,Big Basket,2024-12-21
3,2024-12-31,Flipkart Grocery,2024-12-16
4,2024-12-31,Flipkart National,2024-12-20
...,...,...,...
116,2026-06-30,Flipkart Grocery,2026-06-01
117,2026-06-30,Flipkart National,2026-06-30
118,2026-06-30,Meesho,2026-06-30
119,2026-06-30,Myntra,2026-06-30


In [231]:
chain_wise_max_soh_dates = (
    chain_wise_max_soh_dates
    .groupby('chain')
    .apply(lambda x: dict(zip(x['run_month'], x['as_on_date'])))
    .to_dict()
)

In [232]:
last_date_soh = pd.DataFrame()

for c in chain_wise_max_soh_dates.keys():
    for rm in chain_wise_max_soh_dates[c].keys():
        last_date_soh = pd.concat([
            last_date_soh,
            soh_df[
                (soh_df['chain'] == c) &
                (soh_df['as_on_date'] == chain_wise_max_soh_dates[c][rm])
            ]
        ])

In [233]:
last_date_soh['as_on_date'] = last_date_soh['as_on_date'] + MonthEnd(0)

In [234]:
last_date_soh.groupby(['as_on_date'])['current_soh'].sum()

as_on_date
2024-12-31    246778.066431
2025-01-31    311218.299326
2025-02-28    192679.204553
2025-03-31    244457.303018
2025-04-30    366643.182600
2025-05-31    319795.321935
2025-06-30    398319.051492
2025-07-31    114248.954225
2025-08-31    698712.990596
2025-09-30    774083.078400
2025-10-31    397742.700376
2025-11-30    251355.181461
2025-12-31    435443.790881
2026-01-31    483352.629753
2026-02-28     69373.308492
2026-03-31    282498.423831
2026-04-30    294009.295182
2026-05-31    405360.322214
2026-06-30    490739.067049
Name: current_soh, dtype: float64

In [235]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    last_date_soh[['key', 'as_on_date', 'current_soh']].rename(
        columns={
            'as_on_date': 'month_date',
            'current_soh': 'Actual Closing SOH'
        }
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)

del len_before_merge

In [236]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [237]:
final_df['Actual Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key']
)['Actual Closing SOH'].shift(1)

final_df['Actual Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key']
)['Actual Closing SOH'].shift(2)

In [238]:
final_df['safety_stock'].sum()

3302729.8767983015

In [239]:
final_df['safety_stock'].min()

0.0

In [240]:
# final_df['Assumed Closing SOH'] = np.where(
#     final_df['M month'] == 'M',  
#     final_df['Actual Closing SOH_Lag_1'].fillna(0) + final_df['sec_apo_plan_vol_rum'].fillna(0) \
#     - final_df['Offtake Chain PSKU Forecast Vol'].fillna(0),
#     final_df['safety_stock']
# )

final_df['Assumed Closing SOH'] = final_df['safety_stock']

In [241]:
final_df['Assumed Closing SOH'] = final_df['Assumed Closing SOH'].clip(lower=0.0)

In [242]:
final_df['Assumed Closing SOH_Lag_1'] = final_df.groupby(
    ['run_month', 'key']
)['Assumed Closing SOH'].shift(1)

final_df['Assumed Closing SOH Lag 2'] = final_df.groupby(
    ['run_month', 'key']
)['Assumed Closing SOH'].shift(2)

In [243]:
final_df = final_df[
    ~final_df['material_group_code'].isin(['NC FREE', 'HC FREE'])
]

In [244]:
final_df.isna().sum()

key                                    0
chain                                  0
parent_material_code                   0
material_group_code                    0
run_month                              0
month_date                             0
pri_actuals_vol_rum                18173
sec_apo_plan_vol_rum               18173
Primary P3M                           26
Offtake Chain PSKU                 40267
M month                             6049
Offtake Chain PSKU Forecast Vol    37497
norms_soh                          43421
norm_days                          39664
safety_stock                           0
Actual Closing SOH                 59108
Actual Closing SOH_Lag_1           59108
Actual Closing SOH Lag 2           59108
Assumed Closing SOH                    0
Assumed Closing SOH_Lag_1           6049
Assumed Closing SOH Lag 2          12098
dtype: int64

In [245]:
final_df.shape

(60490, 21)

### Add P3M, LY

In [246]:
actuals_df = plan_actuals_df.copy()
actuals_df['month_date'] = actuals_df['month_date'] + MonthEnd(0)

In [247]:
actuals_df = actuals_df.groupby(
    ['key', 'month_date'], as_index=False
)[['pri_actuals_vol_rum', 'sec_actuals_vol_rum']].sum()

In [248]:
actuals_df.duplicated(subset=['key', 'month_date']).sum()

0

In [249]:
actuals_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [250]:
actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [251]:
actuals_df['Primary P3M redundant'] = actuals_df.groupby(
['key'], as_index = False, group_keys = False)['pri_actuals_vol_rum'].shift(1)\
                            .rolling(window=3, min_periods=1).mean()

In [252]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'Primary Actuals Vol',
        'sec_actuals_vol_rum': 'Sec Actuals Vol'
    }),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [253]:
ly_actuals_df = actuals_df.copy()

In [254]:
ly_actuals_df['month_date'] = ly_actuals_df['month_date'] + MonthEnd(12)

In [255]:
ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M'
    })

,key,month_date,LY Primary Actuals Vol,LY Sec Actuals Vol,LY Primary P3M
0,Amazon ARIPL_715098,2024-07-31,0.0,0.0,NaN
1,Amazon ARIPL_715098,2024-08-31,0.0,0.0,0.0
2,Amazon ARIPL_715098,2024-09-30,0.0,0.0,0.0
3,Amazon ARIPL_715098,2024-10-31,0.0,0.0,0.0
4,Amazon ARIPL_715098,2024-11-30,0.0,0.0,0.0
...,...,...,...,...,...
218486,Nykaa_811416,2027-08-31,0.0,0.0,0.0
218487,Nykaa_811416,2027-09-30,0.0,0.0,0.0
218488,Nykaa_811416,2027-10-31,0.0,0.0,0.0
218489,Nykaa_811416,2027-11-30,0.0,0.0,0.0


In [256]:
ly_actuals_df.sort_values(by=['key', 'month_date'], inplace=True)

In [257]:
ly_actuals_df['pri_actuals_vol_rum_lag_1'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

ly_actuals_df['pri_actuals_vol_rum_lag_2'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

ly_actuals_df['pri_actuals_vol_rum_lag_3'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)


ly_actuals_df['pri_actuals_vol_rum_lead_1'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(-1)

ly_actuals_df['pri_actuals_vol_rum_lead_2'] = ly_actuals_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(-2)

In [258]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_actuals_df.rename(columns={
        'pri_actuals_vol_rum': 'LY Primary Actuals Vol',
        'sec_actuals_vol_rum': 'LY Sec Actuals Vol',
        'Primary P3M redundant': 'LY Primary P3M', 
        'pri_actuals_vol_rum_lag_1': 'LY Primary Actuals Lag 1 Vol',
        'pri_actuals_vol_rum_lag_2': 'LY Primary Actuals Lag 2 Vol',
        'pri_actuals_vol_rum_lag_3': 'LY Primary Actuals Lag 3 Vol',
        'pri_actuals_vol_rum_lead_1': 'LY Primary Actuals Lead 1 Vol',
        'pri_actuals_vol_rum_lead_2': 'LY Primary Actuals Lead 2 Vol',
    }),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [259]:
prev_offtakes_monthly_df = offtakes_monthly_df.copy()

In [260]:
final_offtakes_historical_df = prev_offtakes_monthly_df.copy()

In [261]:
final_offtakes_historical_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()

4

In [262]:
final_offtakes_historical_df['key'] = final_offtakes_historical_df[['chain', 'parent_material_code']].astype(str).agg(
    '_'.join, axis=1
)

In [263]:

final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [264]:
final_offtakes_historical_df['P3M'] = final_offtakes_historical_df.groupby(
    ['key'], as_index = False, group_keys = False)['offtake_vol_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

In [265]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M']].rename(
        columns={'P3M': 'Offtake P3M', 'offtake_vol_rum': 'Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [266]:
ly_final_offtakes_historical_df = final_offtakes_historical_df.copy()

In [267]:

ly_final_offtakes_historical_df.sort_values(
    by=['key', 'month_date'], inplace=True
)

In [268]:
ly_final_offtakes_historical_df['month_date'].min()

Timestamp('2023-01-31 00:00:00')

In [269]:
ly_final_offtakes_historical_df['month_date'] = ly_final_offtakes_historical_df['month_date'] + MonthEnd(12)
ly_final_offtakes_historical_df.head()

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month,P3M
0,2024-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31,NaN
1,2024-02-29,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28,NaN
2,2024-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31,NaN
3,2024-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30,9.020000
4,2024-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31,8.903333


In [270]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lag 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

ly_final_offtakes_historical_df['LY Offtake Actuals Lag 3 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(3)


In [271]:
ly_final_offtakes_historical_df['LY Offtake Actuals Lead 1 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-1)

ly_final_offtakes_historical_df['LY Offtake Actuals Lead 2 Vol'] = ly_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(-2)

In [272]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    ly_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'P3M', 
                                     'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol', 
                                     'LY Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lead 1 Vol',
                                     'LY Offtake Actuals Lead 2 Vol']].rename(
        columns={'P3M': 'LY Offtake P3M', 'offtake_vol_rum': 'LY Offtake Actuals Vol'}
    ),
    on=['key', 'month_date'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [273]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [274]:
for col in ['Primary P3M', 'LY Primary P3M', 'Offtake P3M', 'LY Offtake P3M', 'Primary P3M redundant']:
    # if not 'LY' in col:  'LY P6M',
    final_df.loc[final_df['month_date'] > final_df['run_month'], [col]] = np.nan
    final_df[col] = final_df.groupby(['run_month', 'key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [275]:
final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)

In [276]:
lags_df = plan_actuals_df.copy()

In [277]:
lags_df

,month_date,key,chain,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
24461,2023-07-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
24462,2023-08-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
24463,2023-09-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
24464,2023-10-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
24465,2023-11-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
339972,2026-08-31,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
339973,2026-09-30,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
339974,2026-10-31,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
339975,2026-11-30,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0


In [278]:
lags_df.sort_values(by=['key', 'month_date'], inplace=True)

In [279]:
lags_df

,month_date,key,chain,parent_material_code,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,Primary P3M
24461,2023-07-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
24462,2023-08-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,8.324,0.0,0.0
24463,2023-09-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
24464,2023-10-31,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
24465,2023-11-30,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,0.0,0.0,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
339972,2026-08-31,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
339973,2026-09-30,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
339974,2026-10-31,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0
339975,2026-11-30,Nykaa_811416,Nykaa,811416,SAF-MUSLI,0.0,0.0,0.000,0.0,0.0


In [280]:
lags_df['Primary_Lag_2'] = lags_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

lags_df['Primary_Lag_3'] = lags_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

In [281]:
lags_df['month_date'] = lags_df['month_date'] + MonthEnd(1)

In [282]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_df[['key', 'month_date', 'pri_actuals_vol_rum', 'Primary_Lag_2', 'Primary_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'pri_actuals_vol_rum': 'Primary_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [283]:
lags_final_offtakes_historical_df = final_offtakes_historical_df.copy()

lags_final_offtakes_historical_df.sort_values(by=['key', 'month_date'], inplace=True)
lags_final_offtakes_historical_df['OT_Lag_2'] = lags_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(1)

lags_final_offtakes_historical_df['OT_Lag_3'] = lags_final_offtakes_historical_df.groupby(
    ['key']
)['offtake_vol_rum'].shift(2)

In [284]:
x = lags_final_offtakes_historical_df.copy()
x.rename(columns = {'material_group_code':'Brand'},inplace = True)
len_before_merge = len(x)
x = x.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(x)
del len_before_merge
x['offtake_val_lag3'] = x['OT_Lag_3']*x['Index Rate']/10**7
x['offtake_val_lag2'] = x['OT_Lag_2']*x['Index Rate']/10**7
x['offtake_val_lag1'] = x['offtake_vol_rum']*x['Index Rate']/10**7
x

,month_date,key,chain,parent_material_code,Brand,offtake_vol_rum,run_month,P3M,OT_Lag_2,OT_Lag_3,Index Rate,offtake_val_lag3,offtake_val_lag2,offtake_val_lag1
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31,NaN,NaN,NaN,138865.260689,NaN,NaN,0.133866
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28,NaN,9.640,NaN,138865.260689,NaN,0.133866,0.109912
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31,NaN,7.915,9.640,138865.260689,0.133866,0.109912,0.131991
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30,9.020000,9.505,7.915,138865.260689,0.109912,0.131991,0.129006
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31,8.903333,9.290,9.505,138865.260689,0.131991,0.129006,0.111787
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120070,2026-08-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-08-31,0.000000,0.000,0.000,12860.631072,0.000000,0.000000,0.000000
120071,2026-09-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-09-30,0.000000,0.000,0.000,12860.631072,0.000000,0.000000,0.000000
120072,2026-10-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-10-31,0.000000,0.000,0.000,12860.631072,0.000000,0.000000,0.000000
120073,2026-11-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-11-30,0.000000,0.000,0.000,12860.631072,0.000000,0.000000,0.000000


In [285]:
x.columns
x[x['month_date'] == '2026-06-30'][['offtake_val_lag3', 'offtake_val_lag2',
       'offtake_val_lag1']].sum()

offtake_val_lag3    35.535284
offtake_val_lag2    42.084071
offtake_val_lag1    36.879201
dtype: float64

In [286]:
x[(x['key'].isin(final_df['key'].unique())) & (x['month_date'] == '2026-06-30')][['offtake_val_lag3', 'offtake_val_lag2',
       'offtake_val_lag1']].sum()

offtake_val_lag3    35.179790
offtake_val_lag2    41.767786
offtake_val_lag1    36.617111
dtype: float64

In [287]:
x[(~x['key'].isin(final_df['key'].unique())) & (x['month_date'] == '2026-06-30')].to_csv('key_check.csv')

In [288]:
lags_final_offtakes_historical_df

,month_date,key,chain,parent_material_code,material_group_code,offtake_vol_rum,run_month,P3M,OT_Lag_2,OT_Lag_3
0,2023-01-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.640,2023-01-31,NaN,NaN,NaN
1,2023-02-28,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,7.915,2023-02-28,NaN,9.640,NaN
2,2023-03-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.505,2023-03-31,NaN,7.915,9.640
3,2023-04-30,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,9.290,2023-04-30,9.020000,9.505,7.915
4,2023-05-31,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD,8.050,2023-05-31,8.903333,9.290,9.505
...,...,...,...,...,...,...,...,...,...,...
120118,2026-08-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-08-31,0.000000,0.000,0.000
120119,2026-09-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-09-30,0.000000,0.000,0.000
120120,2026-10-31,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-10-31,0.000000,0.000,0.000
120121,2026-11-30,Purplle_810674,Purplle,810674,PA_ESS_HO,0.000,2026-11-30,0.000000,0.000,0.000


In [289]:
lags_final_offtakes_historical_df['month_date'] = lags_final_offtakes_historical_df['month_date'] + MonthEnd(1)

In [290]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    lags_final_offtakes_historical_df[['key', 'month_date', 'offtake_vol_rum', 'OT_Lag_2', 'OT_Lag_3']].rename(columns={
        'month_date': 'run_month',
        'offtake_vol_rum': 'OT_Lag_1'
    }),
    on=['key', 'run_month'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [291]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60485,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60486,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60487,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60488,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [292]:
x = final_df.copy()
x.rename(columns = {'material_group_code':'Brand'},inplace = True)
len_before_merge = len(x)
x = x.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(x)
del len_before_merge
x['offtake_val'] = x['OT_Lag_2']*x['Index Rate']/10**7
x.groupby(['month_date'])['offtake_val'].sum().reset_index()#[20:]

,month_date,offtake_val
0,2026-06-30,41.767955
1,2026-07-31,41.767955
2,2026-08-31,41.767955
3,2026-09-30,41.767955
4,2026-10-31,41.767955
5,2026-11-30,41.767955
6,2026-12-31,41.767955
7,2027-01-31,41.767955
8,2027-02-28,41.767955
9,2027-03-31,41.767955


In [293]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60485,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60486,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60487,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60488,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [294]:
# final_df.to_csv('ECOM_OTP_v0_check.csv', index=False)

In [295]:
forecasts['run_month'].unique()

<DatetimeArray>
['2026-07-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [296]:
def calculate_primary_iteratively(key_df):
    key_df = key_df.sort_values('month_date').copy()
    key_df = key_df[key_df['month_date'] >= key_df['run_month']]

    # Fill NaNs
    fill_cols = [
        'Offtake Chain PSKU Forecast Vol',
        'safety_stock',
        'Actual Closing SOH_Lag_1'
    ]
    key_df[fill_cols] = key_df[fill_cols].fillna(0)

    _key = key_df['key'].iloc[0]
    _run_month = key_df['run_month'].iloc[0]

    outputs = []
    prev_soh = None

    for _, row in key_df.iterrows():
        forecast = row['Offtake Chain PSKU Forecast Vol']
        safety_stock = row['safety_stock']

        if row['M month'] == 'M':
            opening_soh = row['Actual Closing SOH_Lag_1']
        else:
            opening_soh = prev_soh

        primary_vol = max(
            forecast + safety_stock - opening_soh,
            0
        )

        assumed_closing_soh = max(
            opening_soh + primary_vol - forecast,
            0
        )

        outputs.append({
            'run_month': _run_month,
            'key': _key,
            'month_date': row['month_date'],
            'Calculated Primary Vol': primary_vol,
            'Final Assumed Closing SOH Vol': assumed_closing_soh
        })

        prev_soh = assumed_closing_soh

    return outputs

In [297]:
# # final_df['Calculated Primary Vol'] = np.where(
# #     final_df['M month'] == 'M',
# #     final_df['sec_apo_plan_vol_rum'],
# #     final_df['Offtake Chain PSKU Forecast Vol'].fillna(0) + \
# #         final_df['safety_stock'].fillna(0) - \
# #         final_df['Assumed Closing SOH_Lag_1'].fillna(0)
# # )


# final_df['Calculated Primary Vol'] = final_df['Offtake Chain PSKU Forecast Vol'].fillna(0) + \
#     final_df['safety_stock'].fillna(0) - final_df['Assumed Closing SOH_Lag_1'].fillna(0)

In [298]:
# final_df['Calculated Primary Vol'] = final_df['Calculated Primary Vol'].clip(lower=0)

In [299]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,LY Offtake Actuals Lag 2 Vol,LY Offtake Actuals Lag 3 Vol,LY Offtake Actuals Lead 1 Vol,LY Offtake Actuals Lead 2 Vol,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-07-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-08-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-09-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-10-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60485,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60486,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60487,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-01-31,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
60488,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-02-28,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN


In [300]:
calculated_primary = []

for (_, _), group_df in tqdm(final_df.groupby(['run_month', 'key'])):
    calculated_primary.extend(
        calculate_primary_iteratively(group_df)
    )

calculated_primary_df = pd.DataFrame(calculated_primary)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 6049/6049 [00:13<00:00, 435.95it/s]


In [301]:
del calculated_primary

In [302]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    calculated_primary_df,
    on=['run_month', 'month_date', 'key'],
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [303]:
final_df['Calculated Primary Vol'].min(), final_df['Final Assumed Closing SOH Vol'].min()

(0.0, 0.0)

In [304]:
final_df.sort_values(by=['run_month', 'key', 'month_date'], inplace=True)

In [305]:
final_df['Final Assumed Closing SOH Lag 1 Vol'] = final_df.groupby(
    ['run_month', 'key']
)['Final Assumed Closing SOH Vol'].shift(1)

final_df['Final Assumed Closing SOH Lag 2 Vol'] = final_df.groupby(
    ['run_month', 'key']
)['Final Assumed Closing SOH Vol'].shift(2)

In [306]:
final_df

,key,chain,parent_material_code,material_group_code,run_month,month_date,pri_actuals_vol_rum,sec_apo_plan_vol_rum,Primary P3M,Offtake Chain PSKU,...,Primary_Lag_1,Primary_Lag_2,Primary_Lag_3,OT_Lag_1,OT_Lag_2,OT_Lag_3,Calculated Primary Vol,Final Assumed Closing SOH Vol,Final Assumed Closing SOH Lag 1 Vol,Final Assumed Closing SOH Lag 2 Vol
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-06-30,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-07-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,NaN,NaN
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-08-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,NaN
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-09-30,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,2026-07-31,2026-10-31,0.0,0.0,0.0,NaN,...,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60485,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
60486,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
60487,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-01-31,NaN,NaN,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
60488,Nykaa_811416,Nykaa,811416,SAF-MUSLI,2026-07-31,2027-02-28,NaN,NaN,0.0,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [307]:
brand_md_df

,brand_code,portfolio
0,ADV-AHO-R,Hair Oils
1,ADV-COL-R,Hair Oils
2,ADV-COL-S,Hair Oils
3,BD_BDOL_M,Male Grooming
4,BD_HRWX_M,Male Grooming
...,...,...
391,VEG_CLEAN,Health & Hygiene
392,VEG_CLN_G,Health & Hygiene
393,ZTK DEO,Male Grooming
394,ZTK GODEO,Youth


In [308]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

In [309]:
qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [310]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [311]:
len_before_merge = len(final_df)
final_df = final_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on=['material_group_code'], 
    how='left'
)
assert len_before_merge == len(final_df)
del len_before_merge

In [312]:
final_df.columns

Index(['key', 'chain', 'parent_material_code', 'material_group_code',
       'run_month', 'month_date', 'pri_actuals_vol_rum',
       'sec_apo_plan_vol_rum', 'Primary P3M', 'Offtake Chain PSKU', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'norms_soh', 'norm_days',
       'safety_stock', 'Actual Closing SOH', 'Actual Closing SOH_Lag_1',
       'Actual Closing SOH Lag 2', 'Assumed Closing SOH',
       'Assumed Closing SOH_Lag_1', 'Assumed Closing SOH Lag 2',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol', 'Offtake P3M',
       'LY Offtake Actuals Vol', 'LY Offtake P3M',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol',
       'LY Offtake Actuals Lag 3 Vol', '

In [313]:
final_df = final_df.rename(columns={
    'key': 'Key',
    'chain': 'Chain',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'run_month': 'Run Month',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Till Date Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol',
    'Primary P3M': 'Primary P3M Vol',
    'Offtake Chain PSKU': 'Offtake Chain PSKU Vol',
    'norms_soh': 'Norms SOH',
    'norm_days': 'Norm Days',
    'safety_stock': 'Safety Stock Vol',
    'Actual Closing SOH': 'Actual Closing SOH Vol',
    'Actual Closing SOH_Lag_1': 'Actual Closing SOH Lag 1 Vol',
    'Actual Closing SOH Lag 2': 'Actual Closing SOH Lag 2 Vol',
    'Assumed Closing SOH': 'Assumed Closing SOH Vol',
    'Assumed Closing SOH_Lag_1': 'Assumed Closing SOH Lag 1 Vol',
    'Primary P3M redundant': 'Primary P3M redundant Vol',
    'LY Primary P3M': 'LY Primary P3M Vol',
    'Offtake P3M': 'Offtake P3M Vol',
    'LY Offtake P3M': 'LY Offtake P3M Vol',
    'Primary_Lag_1': 'Primary Actuals Lag 1 Vol',
    'Primary_Lag_2': 'Primary Actuals Lag 2 Vol',
    'Primary_Lag_3': 'Primary Actuals Lag 3 Vol',
    'OT_Lag_1': 'Offtake Actuals Lag 1 Vol',
    'OT_Lag_2': 'Offtake Actuals Lag 2 Vol',
    'OT_Lag_3': 'Offtake Actuals Lag 3 Vol',
    'qtr_ind_rate': 'Index Rate',
    'portfolio': 'Portfolio'
})

In [314]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Offtake Chain PSKU Vol', 'M month',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Assumed Closing SOH Lag 2', 'Primary Actuals Vol', 'Sec Actuals Vol',
       'Primary P3M redundant Vol', 'LY Primary Actuals Vol',
       'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'Offtake Actuals Vol',
       'Offtake P3M Vol', 'LY Offtake Actuals Vol', 'LY Offtake P3M Vol',
       'LY Offtake Actuals Lag 1 Vol', 'LY Offtake Actuals Lag 2 Vol',
       

In [315]:
final_df = final_df[[
    'Key', 'Chain', 'PSKU', 'Brand', 'Index Rate',
    'Portfolio', 'Run Month', 'Month Date',  'M month',

    'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
    'Primary P3M Vol', 'Offtake Chain PSKU Vol',

    'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
    'Safety Stock Vol', 
    
    'Actual Closing SOH Vol', 'Actual Closing SOH Lag 1 Vol', 
    'Actual Closing SOH Lag 2 Vol', 'Assumed Closing SOH Vol',
    'Assumed Closing SOH Lag 1 Vol', 'Final Assumed Closing SOH Vol', 
    'Final Assumed Closing SOH Lag 1 Vol', 
    
    'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
    'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
    'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
    'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
    'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
    'LY Primary Actuals Lead 2 Vol',

    'Offtake Actuals Vol', 'Offtake P3M Vol', 'LY Offtake Actuals Vol',
    'LY Offtake P3M Vol', 
    'Offtake Actuals Lag 1 Vol', 'Offtake Actuals Lag 2 Vol',
    'Offtake Actuals Lag 3 Vol', 'LY Offtake Actuals Lag 1 Vol', 
    'LY Offtake Actuals Lag 2 Vol', 'LY Offtake Actuals Lag 3 Vol', 
    'LY Offtake Actuals Lead 1 Vol', 'LY Offtake Actuals Lead 2 Vol', 

    'Calculated Primary Vol'
]]

In [316]:
vol_to_val_cols = [col for col in final_df.columns if 'Vol' in col]
vol_to_val_cols

['Primary Till Date Actuals Vol',
 'Secondary Plan Vol',
 'Primary P3M Vol',
 'Offtake Chain PSKU Vol',
 'Offtake Chain PSKU Forecast Vol',
 'Safety Stock Vol',
 'Actual Closing SOH Vol',
 'Actual Closing SOH Lag 1 Vol',
 'Actual Closing SOH Lag 2 Vol',
 'Assumed Closing SOH Vol',
 'Assumed Closing SOH Lag 1 Vol',
 'Final Assumed Closing SOH Vol',
 'Final Assumed Closing SOH Lag 1 Vol',
 'Primary Actuals Vol',
 'Sec Actuals Vol',
 'Primary P3M redundant Vol',
 'LY Primary Actuals Vol',
 'LY Sec Actuals Vol',
 'LY Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'Offtake Actuals Vol',
 'Offtake P3M Vol',
 'LY Offtake Actuals Vol',
 'LY Offtake P3M Vol',
 'Offtake Actuals Lag 1 Vol',
 'Offtake Actuals Lag 2 Vol',
 'Offtake Actuals Lag 3 Vol',
 'LY Offtake Actuals L

In [317]:
for col in vol_to_val_cols:
    final_df[col[:-3] + 'Val'] = final_df[col].fillna(0) * final_df['Index Rate'] / (10 ** 7)

In [318]:
final_df

,Key,Chain,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,Primary Till Date Actuals Vol,...,LY Offtake P3M Val,Offtake Actuals Lag 1 Val,Offtake Actuals Lag 2 Val,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val
0,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-07-31,2026-06-30,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-07-31,2026-07-31,M,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-07-31,2026-08-31,M+1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-07-31,2026-09-30,M+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Amazon ARIPL_715098,Amazon ARIPL,715098,CO_SO_PCP,1220.081000,Skin Care,2026-07-31,2026-10-31,M+3,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60485,Nykaa_811416,Nykaa,811416,SAF-MUSLI,315513.490535,Foods,2026-07-31,2026-11-30,M+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60486,Nykaa_811416,Nykaa,811416,SAF-MUSLI,315513.490535,Foods,2026-07-31,2026-12-31,M+5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60487,Nykaa_811416,Nykaa,811416,SAF-MUSLI,315513.490535,Foods,2026-07-31,2027-01-31,M+6,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60488,Nykaa_811416,Nykaa,811416,SAF-MUSLI,315513.490535,Foods,2026-07-31,2027-02-28,M+7,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [319]:
final_df.duplicated(subset=['Run Month', 'Key', 'Month Date']).sum()

0

In [320]:
final_df['Calculated Primary Vol'].sum()

3585232.945111239

### Depot PSKU

#### Aggregate to PSKU first

In [321]:
psku_df = final_df.groupby(
    ['PSKU', 'Brand', 'Portfolio', 'Run Month', 'Month Date'],
    as_index=False
)['Calculated Primary Vol'].sum()

In [322]:
psku_df = psku_df[psku_df['Month Date'] >= psku_df['Run Month']]

In [323]:
psku_df

,PSKU,Brand,Portfolio,Run Month,Month Date,Calculated Primary Vol
1,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-07-31,0.0
2,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-08-31,0.0
3,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-09-30,0.0
4,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-10-31,0.0
5,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-11-30,0.0
...,...,...,...,...,...,...
9305,811416,SAF-MUSLI,Foods,2026-07-31,2026-11-30,0.0
9306,811416,SAF-MUSLI,Foods,2026-07-31,2026-12-31,0.0
9307,811416,SAF-MUSLI,Foods,2026-07-31,2027-01-31,0.0
9308,811416,SAF-MUSLI,Foods,2026-07-31,2027-02-28,0.0


In [324]:
psku_df['Calculated Primary Vol'].sum()

3585232.9451112393

In [325]:
others_df = others_df.groupby(
    ['parent_material_code', 'material_group_code', 'month_date'], as_index=False
)['Primary P3M'].sum()

len_before_merge = len(others_df)
others_df = others_df.merge(
    brand_md_df.rename(
        columns={'brand_code': 'material_group_code'}
    ),
    on='material_group_code',
    how='left'
)
assert len_before_merge == len(others_df)
del len_before_merge

others_df = others_df.rename(columns={
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'Primary P3M': 'Calculated Primary Vol',
    'portfolio': 'Portfolio'
})

In [326]:
others_df

,PSKU,Brand,Month Date,Calculated Primary Vol,Portfolio
0,715098,CO_SO_PCP,2023-07-31,0.0,Skin Care
1,715098,CO_SO_PCP,2023-08-31,0.0,Skin Care
2,715098,CO_SO_PCP,2023-09-30,0.0,Skin Care
3,715098,CO_SO_PCP,2023-10-31,0.0,Skin Care
4,715098,CO_SO_PCP,2023-11-30,0.0,Skin Care
...,...,...,...,...,...
27782,811416,SAF-MUSLI,2026-08-31,0.0,Foods
27783,811416,SAF-MUSLI,2026-09-30,0.0,Foods
27784,811416,SAF-MUSLI,2026-10-31,0.0,Foods
27785,811416,SAF-MUSLI,2026-11-30,0.0,Foods


In [327]:
psku_df['Calculated Primary Vol'].min()

0.0

In [328]:
final_others_forecast = pd.DataFrame()

for rm in psku_df['Run Month'].unique():
    tmp = others_df.copy()
    tmp['Run Month'] = rm
    tmp = tmp[tmp['Month Date'] <= psku_df[psku_df['Run Month'] == rm]['Month Date'].max()]
    tmp = tmp[tmp['Month Date'] >= rm]
    final_others_forecast = pd.concat([final_others_forecast, tmp], ignore_index=True)
    del tmp

In [329]:
psku_df = pd.concat(
    [psku_df, final_others_forecast], ignore_index=True
)

In [330]:
psku_df = psku_df.groupby(
    ['PSKU', 'Brand', 'Portfolio', 'Run Month', 'Month Date'],
    as_index=False
)['Calculated Primary Vol'].sum()

In [331]:
psku_df['Calculated Primary Vol'].sum()

3603601.0191112403

In [332]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)

In [333]:
depot_psku_primary_df

,DEPOT_CODE,PARENT_MATERIAL_CODE,MATERIAL_GROUP_CODE,MONTH_DATE,PRI_ACTUALS_VOL_RUM,PRI_APO_PLAN_VOL_RUM,SEC_APO_PLAN_VOL_RUM,SEC_ACTUALS_VOL_RUM
0,D111,718471,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
1,D111,718471,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
2,D111,718472,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
3,D111,718472,ADV-AHO-R,2017-07-31,0.0,0.000000,NaN,0.0
4,D111,718473,ADV-AHO-R,2017-04-30,0.0,0.000000,NaN,0.0
...,...,...,...,...,...,...,...,...
336356,D677,810179,SW_SGPRF,2026-05-31,0.0,19.109470,0.0,0.0
336357,D677,810179,SW_SGPRF,2026-06-30,0.0,21.831578,0.0,0.0
336358,D677,810179,SW_SGPRF,2026-07-31,0.0,7.823442,0.0,0.0
336359,D677,811169,SW_SGPRF,2026-03-31,0.0,9.818184,0.0,0.0


In [334]:
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [335]:
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')

In [336]:
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()

In [337]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]

In [338]:
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()

147

In [339]:
depot_psku_primary_df.shape

(329302, 8)

In [340]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
depot_psku_primary_df = depot_psku_primary_df.drop(indices_to_drop)

# Verify no duplicates remain
print(depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum())

0


In [341]:
depot_psku_primary_df.shape

(329098, 8)

In [342]:
depot_psku_primary_df = impute_missing_dates(
    depot_psku_primary_df.copy(),
    key=['depot_code', 'parent_material_code'],
    date_col='month_date'
)

18949it [00:04, 4294.37it/s]


In [343]:
cols = ['depot_code', 'parent_material_code', 'material_group_code']

depot_psku_primary_df[cols] = depot_psku_primary_df.groupby('key')[cols].transform(lambda x: x.ffill().bfill())

In [344]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].fillna(0)

In [345]:
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)

In [346]:
depot_psku_primary_df.duplicated(subset=['key', 'month_date']).sum()


0

In [347]:
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)



pri_actuals_vol_rum
sec_actuals_vol_rum


In [348]:
depot_psku_primary_df.sort_values(by=['key', 'month_date'], inplace=True)

In [349]:
depot_psku_primary_df['Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key'], 
    as_index = False, group_keys = False
)['pri_actuals_vol_rum'].shift(1).rolling(window=3, min_periods=1).mean()

In [350]:
depot_psku_primary_df['Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(1)

depot_psku_primary_df['Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(2)

depot_psku_primary_df['Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(3)

depot_psku_primary_df['LY Primary Actuals Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(12)

depot_psku_primary_df['LY Primary Actuals Lag 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(13)

depot_psku_primary_df['LY Primary Actuals Lag 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(14)

depot_psku_primary_df['LY Primary Actuals Lag 3 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(15)


depot_psku_primary_df['LY Primary Actuals Lead 1 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(11)

depot_psku_primary_df['LY Primary Actuals Lead 2 Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['pri_actuals_vol_rum'].shift(10)

In [351]:
depot_psku_primary_df['LY Primary P3M Vol'] = depot_psku_primary_df.groupby(
    ['key']
)['Primary P3M Vol'].shift(12)

In [352]:
depot_psku_primary_df = depot_psku_primary_df.rename(columns={
    'key': 'Key',
    'depot_code': 'Depot',
    'parent_material_code': 'PSKU',
    'material_group_code': 'Brand',
    'month_date': 'Month Date',
    'pri_actuals_vol_rum': 'Primary Actuals Vol',
    'pri_apo_plan_vol_rum': 'Primary Plan Vol',
    'sec_actuals_vol_rum': 'Secondary Actuals Vol',
    'sec_apo_plan_vol_rum': 'Secondary Plan Vol'
})

In [353]:
depot_psku_primary_df

,Month Date,Key,Depot,PSKU,Brand,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,Secondary Actuals Vol,Primary P3M Vol,Primary Actuals Lag 1 Vol,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol
0,2017-04-30,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-05-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-06-30,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-07-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-08-31,D111_709567,D111,709567,SAFF OATS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962499,2026-08-31,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
962500,2026-09-30,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
962501,2026-10-31,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
962502,2026-11-30,D677_811416,D677,811416,SAF-MUSLI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [354]:
final_depot_psku_df = depot_psku_primary_df[['Key', 'Depot', 'PSKU', 'Brand']].drop_duplicates()

In [355]:
psku_df['Calculated Primary Vol'].sum()

3603601.0191112403

In [356]:
x = psku_df.copy()

In [357]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = psku_df[psku_df.duplicated(subset=['PSKU','Run Month', 'Month Date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['Brand'] != 'PABABY_ML'].index

# Remove those rows
psku_df = psku_df.drop(indices_to_drop)

# Verify no duplicates remain
print(psku_df.duplicated(subset=['PSKU','Run Month', 'Month Date']).sum())

0


In [358]:
psku_df['Calculated Primary Vol'].sum()

3602999.9091434986

In [359]:
tmp_df = pd.DataFrame()

for rm in ['2026-07-31']: 
    mth_dates = [pd.to_datetime(rm) + MonthEnd(i) for i in range(-1, 9)]
    for mth_dt in mth_dates:
        tmp_df2 = final_depot_psku_df.copy()
        tmp_df2['Run Month'] = pd.to_datetime(rm)
        tmp_df2['Month Date'] = pd.to_datetime(mth_dt)

        tmp_df = pd.concat([tmp_df, tmp_df2], ignore_index=True)
        del tmp_df2

final_depot_psku_df = tmp_df.copy()    
del tmp_df

In [360]:
psku_df

,PSKU,Brand,Portfolio,Run Month,Month Date,Calculated Primary Vol
0,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-07-31,0.0
1,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-08-31,0.0
2,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-09-30,0.0
3,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-10-31,0.0
4,715098,CO_SO_PCP,Skin Care,2026-07-31,2026-11-30,0.0
...,...,...,...,...,...,...
8446,811416,SAF-MUSLI,Foods,2026-07-31,2026-11-30,0.0
8447,811416,SAF-MUSLI,Foods,2026-07-31,2026-12-31,0.0
8448,811416,SAF-MUSLI,Foods,2026-07-31,2027-01-31,0.0
8449,811416,SAF-MUSLI,Foods,2026-07-31,2027-02-28,0.0


In [361]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    psku_df[['PSKU', 'Run Month', 'Month Date', 'Calculated Primary Vol']], 
    on=['PSKU', 'Run Month', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [364]:
final_depot_psku_df['Calculated Primary Vol'].sum()

83120839.18255588

In [368]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN
1,D111_710981,D111,710981,PA_MEN_AH,2026-07-31,2026-06-30,NaN
2,D111_711040,D111,711040,PCNO GOLD,2026-07-31,2026-06-30,NaN
3,D111_711041,D111,711041,PCNO GOLD,2026-07-31,2026-06-30,NaN
4,D111_711042,D111,711042,PCNO GOLD,2026-07-31,2026-06-30,NaN
...,...,...,...,...,...,...,...
190585,D677_811268,D677,811268,PA_ESS_HO,2026-07-31,2027-03-31,0.993323
190586,D677_811269,D677,811269,PA_ESS_HO,2026-07-31,2027-03-31,0.029355
190587,D677_811279,D677,811279,SAF_CDPRS,2026-07-31,2027-03-31,0.000000
190588,D677_811287,D677,811287,PA_RSW_SR,2026-07-31,2027-03-31,NaN


In [371]:
x = final_depot_psku_df.groupby(['PSKU','Month Date'])['Calculated Primary Vol'].mean().reset_index()
x[x['Month Date']>'2026-06-30']['Calculated Primary Vol'].sum()

3602999.909143498

In [372]:
depot_psku_primary_df.columns

Index(['Month Date', 'Key', 'Depot', 'PSKU', 'Brand', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol'],
      dtype='object')

In [373]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    depot_psku_primary_df[['Depot', 'PSKU', 'Month Date', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',  'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol', 
       'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol']], 
    on=['Depot', 'PSKU', 'Month Date'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [374]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 1 Vol,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190585,D677_811268,D677,811268,PA_ESS_HO,2026-07-31,2027-03-31,0.993323,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190586,D677_811269,D677,811269,PA_ESS_HO,2026-07-31,2027-03-31,0.029355,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190587,D677_811279,D677,811279,SAF_CDPRS,2026-07-31,2027-03-31,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190588,D677_811287,D677,811287,PA_RSW_SR,2026-07-31,2027-03-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [376]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol'],
      dtype='object')

In [377]:
final_depot_psku_df['Primary P3M copy Vol'] = final_depot_psku_df['Primary P3M Vol'].copy()

In [378]:
final_depot_psku_df['Primary P3M Vol'] = final_depot_psku_df['Primary P3M Vol'].fillna(0)

In [379]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 2 Vol,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190585,D677_811268,D677,811268,PA_ESS_HO,2026-07-31,2027-03-31,0.993323,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190586,D677_811269,D677,811269,PA_ESS_HO,2026-07-31,2027-03-31,0.029355,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190587,D677_811279,D677,811279,SAF_CDPRS,2026-07-31,2027-03-31,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190588,D677_811287,D677,811287,PA_RSW_SR,2026-07-31,2027-03-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [380]:
final_depot_psku_df['PSKU Primary P3M Sum Vol'] = final_depot_psku_df.groupby(
    ['Run Month', 'Month Date', 'PSKU']
)['Primary P3M Vol'].transform('sum')

In [381]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,D111_710981,D111,710981,PA_MEN_AH,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,D111_711040,D111,711040,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,D111_711041,D111,711041,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,D111_711042,D111,711042,PCNO GOLD,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190585,D677_811268,D677,811268,PA_ESS_HO,2026-07-31,2027-03-31,0.993323,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
190586,D677_811269,D677,811269,PA_ESS_HO,2026-07-31,2027-03-31,0.029355,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
190587,D677_811279,D677,811279,SAF_CDPRS,2026-07-31,2027-03-31,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
190588,D677_811287,D677,811287,PA_RSW_SR,2026-07-31,2027-03-31,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [382]:
final_depot_psku_df.sort_values(by=['Run Month', 'Key', 'Month Date'], inplace=True)

In [383]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol',
       'Primary P3M copy Vol', 'PSKU Primary P3M Sum Vol'],
      dtype='object')

In [384]:
for col in ['Primary P3M Vol', 'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol', 
            'Primary Actuals Lag 3 Vol', 'PSKU Primary P3M Sum Vol']:
    # if not 'LY' in col:  'LY P6M',
    final_depot_psku_df.loc[final_depot_psku_df['Month Date'] > final_depot_psku_df['Run Month'], [col]] = np.nan
    final_depot_psku_df[col] = final_depot_psku_df.groupby(['Run Month', 'Key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [385]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
19059,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
38118,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
57177,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
76236,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-10-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114353,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.2815
133412,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.2815
152471,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.2815
171530,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2027-02-28,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.2815


In [386]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated Primary Vol', 'Primary Actuals Vol', 'Primary Plan Vol',
       'Secondary Plan Vol', 'Secondary Actuals Vol', 'Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Vol',
       'LY Primary Actuals Lag 1 Vol', 'LY Primary Actuals Lag 2 Vol',
       'LY Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lead 1 Vol',
       'LY Primary Actuals Lead 2 Vol', 'LY Primary P3M Vol',
       'Primary P3M copy Vol', 'PSKU Primary P3M Sum Vol'],
      dtype='object')

In [387]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Vol,LY Primary Actuals Vol,LY Primary Actuals Lag 1 Vol,LY Primary Actuals Lag 2 Vol,LY Primary Actuals Lag 3 Vol,LY Primary Actuals Lead 1 Vol,LY Primary Actuals Lead 2 Vol,LY Primary P3M Vol,Primary P3M copy Vol,PSKU Primary P3M Sum Vol
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
19059,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
38118,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
57177,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
76236,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-10-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114353,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.2815
133412,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2.2815
152471,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.2815
171530,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2027-02-28,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.2815


In [389]:
final_depot_psku_df['PSKU P3M Contribution'] = final_depot_psku_df['Primary P3M Vol'] / final_depot_psku_df['PSKU Primary P3M Sum Vol']

In [394]:
final_depot_psku_df['PSKU P3M Contribution'].isnull().sum()

110190

In [ ]:
x = final_depot_psku_df.groupby(['PSKU','Month Date'])['Calculated Primary Vol'].mean().reset_index()
x[x['Month Date']>'2026-06-30']['Calculated Primary Vol'].sum()

In [390]:
final_depot_psku_df['Calculated Depot PSKU Primary Vol'] = final_depot_psku_df['Calculated Primary Vol'] \
    * final_depot_psku_df['PSKU P3M Contribution']

In [392]:
final_depot_psku_df.rename(columns={'Calculated Primary Vol': 'Calculated PSKU Primary Vol'}, inplace=True)

In [393]:
final_depot_psku_df['Calculated Depot PSKU Primary Vol'].sum()

3559783.006992634

In [379]:
vol_to_val_cols = [col for col in final_depot_psku_df.columns if ('Vol' in col) and ('copy' not in col)]
vol_to_val_cols

['Calculated PSKU Primary Vol',
 'Primary Actuals Vol',
 'Primary Plan Vol',
 'Secondary Plan Vol',
 'Secondary Actuals Vol',
 'Primary P3M Vol',
 'Primary Actuals Lag 1 Vol',
 'Primary Actuals Lag 2 Vol',
 'Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Vol',
 'LY Primary Actuals Lag 1 Vol',
 'LY Primary Actuals Lag 2 Vol',
 'LY Primary Actuals Lag 3 Vol',
 'LY Primary Actuals Lead 1 Vol',
 'LY Primary Actuals Lead 2 Vol',
 'LY Primary P3M Vol',
 'PSKU Primary P3M Sum Vol',
 'Calculated Depot PSKU Primary Vol']

In [380]:
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [381]:
len_before_merge = len(final_depot_psku_df)
final_depot_psku_df = final_depot_psku_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(columns={
        'brand_code': 'Brand',
        'qtr_ind_rate': 'Index Rate'
    }),
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(final_depot_psku_df)
del len_before_merge

In [382]:
for col in vol_to_val_cols:
    final_depot_psku_df[col[:-3] + 'Val'] = final_depot_psku_df[col] * final_depot_psku_df['Index Rate'] / (10 ** 7)

In [383]:
final_depot_psku_df

,Key,Depot,PSKU,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,Secondary Plan Vol,...,Primary Actuals Lag 3 Val,LY Primary Actuals Val,LY Primary Actuals Lag 1 Val,LY Primary Actuals Lag 2 Val,LY Primary Actuals Lag 3 Val,LY Primary Actuals Lead 1 Val,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val
0,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-06-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,NaN
1,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-07-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,NaN
2,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-08-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,NaN
3,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-09-30,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,NaN
4,D111_709567,D111,709567,SAFF OATS,2026-07-31,2026-10-31,NaN,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190585,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2026-11-30,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.071984,0.0
190586,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2026-12-31,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.071984,0.0
190587,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2027-01-31,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.071984,0.0
190588,D677_811416,D677,811416,SAF-MUSLI,2026-07-31,2027-02-28,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.071984,0.0


In [384]:
final_depot_psku_df['Calculated Depot PSKU Primary Val'] = final_depot_psku_df['Calculated Depot PSKU Primary Val'].fillna(0)

In [385]:
final_depot_psku_df['M Month'] = final_depot_psku_df.apply(
    lambda x: mappings[x['Run Month']].get(x['Month Date'], np.nan),
    axis=1
)

In [386]:
final_depot_psku_df = final_depot_psku_df[final_depot_psku_df['M Month'].notna()]

In [387]:
final_depot_psku_df.groupby(
    ['Run Month', 'Month Date', 'PSKU']
)['PSKU P3M Contribution'].max().max()

1.0

In [388]:
final_df.columns

Index(['Key', 'Chain', 'PSKU', 'Brand', 'Index Rate', 'Portfolio', 'Run Month',
       'Month Date', 'M month', 'Primary Till Date Actuals Vol',
       'Secondary Plan Vol', 'Primary P3M Vol', 'Offtake Chain PSKU Vol',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'Off

In [389]:
final_df['Calculated Primary Val'].sum(), final_depot_psku_df['Calculated Depot PSKU Primary Val'].sum()

(363.8818775948206, 358.51868964509106)

In [396]:
final_df[final_df['Month Date']=='2026-08-31']['Calculated Primary Val'].sum()

52.57898868168692

In [391]:
final_df['Chain'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa'], dtype=object)

In [392]:
final_df[final_df['Month Date']=='2026-08-31']['Offtake Actuals Lag 1 Val'].sum()

36.61711124603579

In [393]:
final_df['Chain'].unique()

array(['Amazon ARIPL', 'Amazon RK', 'Big Basket', 'Flipkart Grocery',
       'Flipkart National', 'Meesho', 'Myntra', 'Nykaa'], dtype=object)

In [394]:
final_depot_psku_df[final_depot_psku_df['Month Date']=='2026-08-31']['LY Primary Actuals Val'].sum()

61.148358665079606

In [395]:
final_depot_psku_df.columns

Index(['Key', 'Depot', 'PSKU', 'Brand', 'Run Month', 'Month Date',
       'Calculated PSKU Primary Vol', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol', 'Primary P3M copy Vol',
       'PSKU Primary P3M Sum Vol', 'PSKU P3M Contribution',
       'Calculated Depot PSKU Primary Vol', 'Index Rate',
       'Calculated PSKU Primary Val', 'Primary Actuals Val',
       'Primary Plan Val', 'Secondary Plan Val', 'Secondary Actuals Val',
       'Primary P3M Val', 'Primary Actuals Lag 1 Val',
       'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
       'LY Primary Actuals Val', 'LY Primar

## SAVE

In [397]:
VERSION = 'Jul26 Live Run' 

In [1094]:
os.makedirs(f'FINAL ECOM Chain PSKU OTP/{VERSION}')

FileExistsError: [Errno 17] File exists: 'FINAL ECOM Chain PSKU OTP/Jul26 Live Run'

In [398]:
final_depot_psku_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/ECOM Depot PSKU Primary_{VERSION}.csv', index=False)
final_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/ECOM Chain PSKU Primary_{VERSION}.csv', index=False)

In [311]:
norm_days.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/Norm Days {VERSION}.csv', index=False)
norms.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/Norms {VERSION}.csv', index=False)

In [312]:
soh_df.to_csv(f'FINAL ECOM Chain PSKU OTP/{VERSION}/SOH {VERSION}.csv', index=False)